This notebook was run on a Colab kernel.

# Setup

---
---

**Mount Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Verify Folder structure in drive**

In [ ]:
import os
from collections import Counter

def verify_dataset(base_paths):
    print("VERIFYING DATASET STRUCTURE...")
    print("=" * 60)

    total_dataset_size = 0
    total_dataset_files = 0

    for base_path in base_paths:
        if not os.path.exists(base_path):
            print(f"Not found: {base_path} (Skipping)")
            continue

        for root, dirs, files in os.walk(base_path):
            folder_size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
            total_dataset_size += folder_size
            total_dataset_files += len(files)

            ext_counts = Counter(os.path.splitext(f)[1].lower() for f in files)

            level = root.replace(base_path, '').count(os.sep)
            indent = ' ' * 4 * level
            folder_name = os.path.basename(root) if root != base_path else os.path.basename(base_path)

            print(f"{indent} {folder_name}/  [{len(files)} files | {folder_size / (1024*1024):.2f} MB]")

            if files:
                for ext, count in ext_counts.items():
                    ext_name = ext if ext else "no_extension"
                    print(f"{indent}    ├── {count} {ext_name} files")

                examples = sorted(files)
                if len(files) >= 3:
                    examples_str = f"{examples[0]}, {examples[1]}, {examples[2]}, ..."
                else:
                    examples_str = ", ".join(examples)
                print(f"{indent}    └── Examples: {examples_str}\n")
            else:
                print(f"{indent}    └── (Empty folder)\n")

    print("=" * 60)
    print(f"Total Dataset Files: {total_dataset_files}")
    print(f"Total Dataset Size:  {total_dataset_size / (1024*1024):.2f} MB")

paths_to_check = [
    '/content/drive/MyDrive/pheno_crop',
]

verify_dataset(paths_to_check)

# we don't need the sentinel_2_mixed_temp folder .. ignore it for now

VERIFYING DATASET STRUCTURE...
 pheno_crop/  [1 files | 0.07 MB]
    ├── 1 .csv files
    └── Examples: ground_truth.csv

     sentinel_1/  [52 files | 0.17 MB]
        ├── 52 .csv files
        └── Examples: plot_10_sar.csv, plot_11_sar.csv, plot_12_sar.csv, ...

     sentinel_2_mixed_temp/  [239 files | 0.92 MB]
        ├── 227 .tif files
        ├── 12 .csv files
        └── Examples: plot_10_indices.csv, plot_11_indices.csv, plot_12_indices.csv, ...

     sentinel_2/  [52 files | 0.95 MB]
        ├── 52 .csv files
        └── Examples: plot_10_indices.csv, plot_11_indices.csv, plot_12_indices.csv, ...

Total Dataset Files: 344
Total Dataset Size:  2.11 MB


---
---

# Models

***

## Dual-Stream Hybrid Crop Phenology Network
**Architecture Blueprint**

This model is a late-fusion, multimodal neural network designed to predict winter wheat growth stages (Stages 0-4) using asynchronous, cloud-obstructed satellite imagery. It processes two independent data streams: Sentinel-2 (Optical) and Sentinel-1 (Radar); before fusing their latent representations into a final predictive multi-layer perceptron (MLP).

### Phase 1: The Asynchronous Data Pipeline (Alignment)
Standard neural networks require periodic, rigid time series. Satellite flyovers are highly irregular, and optical data is frequently corrupted by clouds. The dataset class acts as the alignment engine to solve this.

* **The Anchor Point:** The exact date the ground truth observation was made is set as `Day 0`. The dataset looks backward 90 days.
* **Sequence Padding:**  The pipeline extracts a maximum of 30 satellite flyovers within that window. If a field has fewer than 30 flyovers, the remaining slots are padded with zeros.
* **The Time Tensor:** For every reading, the pipeline calculates the exact `days_ago` integer. 
* **The Outputs:** For each field, the dataloader produces four parallel tensors:
  * Optical: `s2_feats` shape $30 \times 9$, `s2_days` shape $30$
  * Radar: `s1_feats` shape $30 \times 3$, `s1_days` shape $30$

---

### Phase 2A: The Sentinel-2 (Optical) Pipeline 
**Design:** CNN + Transformer Encoder Hybrid 

**Purpose:** To extract the biological growth curve while dynamically filtering out cloudy/corrupted days.

1. **Local Micro-Trend Extraction (1D CNN):** The $30 \times 9$ optical tensor passes through a 1D Convolutional Neural Network with a 3-day sliding window. This smooths out sensor noise and calculates local gradients (e.g., a sharp 3-day drop in moisture), projecting the features into a richer $64$-D space.
2. **Positional Time Embedding:** The integer `s2_days` tensor is passed through an `nn.Embedding(120, 64)`. This creates a $64$-D mathematical signature representing *exactly when* the reading occurred. This is added to the CNN output, ensuring the network understands the massive irregular gaps between clear days.
3. **The `[CLS]` Token Injection:**  A blank, learnable $64$-D vector (the Classification Token) is prepended to the 30-day sequence, expanding the sequence length to 31. This token acts as a sponge.
4. **Global Reasoning (Transformer Stack):** The 31-step sequence enters a stack of Transformer Encoder blocks. The Multi-Head Attention mechanism looks at the entire 90-day timeline simultaneously. It mathematically mutes days with high `cloud_pct` and maps the long-term relationships of the vegetation indices. 
5. **Output Slice:** The network discards the 30 real days and outputs only the heavily refined `[CLS]` token, yielding a single $64$-D summary vector of the optical history.

---

### Phase 2B: The Sentinel-1 (Radar) Pipeline
**Design:** CNN + GRU Hybrid

**Purpose:** To track the continuous physical structure and volume scattering of the crop canopy.

1. **Noise Reduction & Shape Detection (1D CNN):** Radar data is inherently noisy (speckle). The $30 \times 3$ radar tensor passes through a 1D CNN. The sliding window smooths the speckle and identifies short-term structural shapes (like the steep drop in VH backscatter right before harvest). It projects the 3 features into a $64$-D space.
2. **Positional Time Embedding:** Just like the optical stream, the `s1_days` integers are embedded and added to the CNN features, anchoring the structural shapes to their exact place in the 90-day window.
3. **Long-Term Structural Memory (GRU):**  Because radar effortlessly punches through clouds, the timeline is highly continuous. The time-stamped CNN features are fed sequentially into a Gated Recurrent Unit (GRU). The GRU steps through the days, smoothly updating its internal "memory" of how tall and dense the wheat stalks are getting.
4. **Output State:** The network extracts the final hidden state of the GRU, yielding a single $64$-D summary vector of the field's structural history.

---

### Phase 3: The Late Fusion Layer

Up until this point, the two pipelines have operated completely independently. The Fusion Layer bridges the physics and the chemistry.
* The $64$-D Optical summary and the $64$-D Radar summary are concatenated into a single $128$-D vector.

### Phase 4: The Multi-Layer Perceptron (Classifier)
The fused $128$-D vector is passed through a dense Feed-Forward network.
* **Architecture:** `Linear(128, 64) -> BatchNorm1d -> ReLU -> Dropout(0.4) -> Linear(64, 5)`
* **Dropout & BatchNorm:** Included to prevent the model from overfitting to the training plots, forcing it to learn robust, generalized rules.
* **Output:** The final Linear layer outputs 5 raw logits representing the probabilities for Stage 0, Stage 1, Stage 2, Stage 3, and Stage 4.

---

#### Setting up the Dataset

In [4]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

class ProductionPhenologyDataset(Dataset):
    def __init__(self, gt_path, s1_dir, s2_dir, plot_ids, lookback_days=90, max_seq_len=30):
        self.lookback_days = lookback_days
        self.max_seq_len = max_seq_len
        
        # Load Ground Truth
        self.gt = pd.read_csv(gt_path)
        self.gt['date'] = pd.to_datetime(self.gt['date'])
        self.gt = self.gt[self.gt['plot_id'].isin(plot_ids)].reset_index(drop=True)
        
        self.s1_data, self.s2_data = {}, {}
        
        print("Loading CSVs into memory...")
        for pid in plot_ids:
            # Load Radar
            s1_path = os.path.join(s1_dir, f"plot_{pid}_sar.csv")
            if os.path.exists(s1_path):
                df1 = pd.read_csv(s1_path)
                df1['date'] = pd.to_datetime(df1['date'])
                self.s1_data[pid] = df1
                
            # Load Optical
            s2_path = os.path.join(s2_dir, f"plot_{pid}_indices.csv")
            if os.path.exists(s2_path):
                df2 = pd.read_csv(s2_path)
                df2['date'] = pd.to_datetime(df2['date'])
                self.s2_data[pid] = df2
        print(f"Dataset ready. Found {len(self.gt)} ground truth events.")

    def __len__(self): return len(self.gt)

    def __getitem__(self, idx):
        row = self.gt.iloc[idx]
        target_date, plot_id, label = row['date'], row['plot_id'], row['stage_code']
        start_date = target_date - pd.Timedelta(days=self.lookback_days)
        
        # --- RADAR TENSORS (S1) ---
        s1_df = self.s1_data.get(plot_id, pd.DataFrame())
        s1_tensor = torch.zeros((self.max_seq_len, 3), dtype=torch.float32)
        s1_days_tensor = torch.zeros((self.max_seq_len,), dtype=torch.float32)
        
        if not s1_df.empty:
            mask = (s1_df['date'] > start_date) & (s1_df['date'] <= target_date)
            valid_s1 = s1_df[mask]
            if len(valid_s1) > 0:
                features = valid_s1[['VV', 'VH', 'VH_VV_Ratio']].values
                days_ago = (target_date - valid_s1['date']).dt.days.values
                seq_len = min(len(features), self.max_seq_len)
                s1_tensor[-seq_len:] = torch.tensor(features[-seq_len:], dtype=torch.float32)
                s1_days_tensor[-seq_len:] = torch.tensor(days_ago[-seq_len:], dtype=torch.float32)

        # --- OPTICAL TENSORS (S2) ---
        s2_df = self.s2_data.get(plot_id, pd.DataFrame())
        s2_tensor = torch.zeros((self.max_seq_len, 9), dtype=torch.float32)
        s2_days_tensor = torch.zeros((self.max_seq_len,), dtype=torch.float32)
        
        if not s2_df.empty:
            mask = (s2_df['date'] > start_date) & (s2_df['date'] <= target_date)
            valid_s2 = s2_df[mask]
            if len(valid_s2) > 0:
                features = valid_s2[['NDVI', 'NDWI', 'NDRE', 'EVI', 'SAVI', 'MSAVI', 'NDMI', 'GNDVI', 'cloud_pct']].values
                days_ago = (target_date - valid_s2['date']).dt.days.values
                seq_len = min(len(features), self.max_seq_len)
                s2_tensor[-seq_len:] = torch.tensor(features[-seq_len:], dtype=torch.float32)
                s2_days_tensor[-seq_len:] = torch.tensor(days_ago[-seq_len:], dtype=torch.float32)

        return {
            's1_feats': s1_tensor, 's1_days': s1_days_tensor,
            's2_feats': s2_tensor, 's2_days': s2_days_tensor,
            'label': torch.tensor(label, dtype=torch.long)
        }

#### Model Setup

In [5]:
# ---------------------------------------------------------
# STREAM A: Advanced Optical Branch (CNN + Transformer + CLS)
# ---------------------------------------------------------
class AdvancedOpticalBranch(nn.Module):
    def __init__(self, in_features=9, embed_dim=64, num_layers=2):
        super().__init__()
        # Micro-trend smoothing
        self.local_cnn = nn.Sequential(
            nn.Conv1d(in_channels=in_features, out_channels=embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=embed_dim, out_channels=embed_dim, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.time_embed = nn.Embedding(120, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=128, dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x, days_ago):
        batch_size = x.size(0)
        # CNN Phase
        x = x.transpose(1, 2) 
        cnn_out = self.local_cnn(x).transpose(1, 2) 
        
        # Time Phase
        days_clamped = torch.clamp(days_ago.long(), 0, 119)
        feats_with_time = cnn_out + self.time_embed(days_clamped)
        
        # Transformer & CLS Phase
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        transformer_in = torch.cat((cls_tokens, feats_with_time), dim=1)
        transformer_out = self.transformer(transformer_in)
        
        # Return only the CLS sponge
        return transformer_out[:, 0, :]

# ---------------------------------------------------------
# STREAM B: Advanced Radar Branch (CNN + GRU)
# ---------------------------------------------------------
class AdvancedRadarBranch(nn.Module):
    def __init__(self, in_features=3, embed_dim=64):
        super().__init__()
        # Noise reduction & local shapes
        self.local_cnn = nn.Sequential(
            nn.Conv1d(in_channels=in_features, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=embed_dim, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.time_embed = nn.Embedding(120, embed_dim)
        # Recurrent memory for continuous structural growth
        self.gru = nn.GRU(input_size=embed_dim, hidden_size=embed_dim, num_layers=2, batch_first=True, dropout=0.2)

    def forward(self, x, days_ago):
        # CNN Phase
        x = x.transpose(1, 2)
        cnn_out = self.local_cnn(x).transpose(1, 2)
        
        # Time Phase
        days_clamped = torch.clamp(days_ago.long(), 0, 119)
        feats_with_time = cnn_out + self.time_embed(days_clamped)
        
        # GRU Phase (we only want the final hidden state)
        _, h_n = self.gru(feats_with_time)
        # h_n shape is [num_layers, batch, hidden_size]. We take the top layer's output.
        return h_n[-1]

# ---------------------------------------------------------
# THE FUSION LAYER: Dual-Stream Phenology Net
# ---------------------------------------------------------
class MasterPhenologyNet(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.optical_branch = AdvancedOpticalBranch(in_features=9, embed_dim=64)
        self.radar_branch = AdvancedRadarBranch(in_features=3, embed_dim=64)
        
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, s1_feats, s1_days, s2_feats, s2_days):
        radar_summary = self.radar_branch(s1_feats, s1_days)
        optical_summary = self.optical_branch(s2_feats, s2_days)
        
        fused = torch.cat([radar_summary, optical_summary], dim=1)
        return self.classifier(fused)

#### Training

In [ ]:
def train_production_model(dataset, epochs=50, batch_size=32, lr=0.001):
    # Setup Split (80% Train, 20% Val)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    # Calculate Weights using Training Data Only
    print("Calculating dynamic class weights...")
    train_labels = [train_ds[i]['label'].item() for i in range(len(train_ds))]
    class_counts = torch.bincount(torch.tensor(train_labels), minlength=5)
    class_counts = torch.where(class_counts == 0, torch.tensor(1), class_counts)
    class_weights = len(train_labels) / (5.0 * class_counts.float())
    
    # Device setup (CPU is perfectly fine for these CSVs, but GPU ready just in case)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device} | {train_size} Train events | {val_size} Val events")
    
    model = MasterPhenologyNet(num_classes=5).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    for epoch in range(epochs):
        # --- TRAINING ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for batch in train_loader:
            s1_f, s1_d = batch['s1_feats'].to(device), batch['s1_days'].to(device)
            s2_f, s2_d = batch['s2_feats'].to(device), batch['s2_days'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            logits = model(s1_f, s1_d, s2_f, s2_d)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * labels.size(0)
            _, preds = torch.max(logits, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
        # --- VALIDATION ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                s1_f, s1_d = batch['s1_feats'].to(device), batch['s1_days'].to(device)
                s2_f, s2_d = batch['s2_feats'].to(device), batch['s2_days'].to(device)
                labels = batch['label'].to(device)
                
                logits = model(s1_f, s1_d, s2_f, s2_d)
                loss = criterion(logits, labels)
                
                val_loss += loss.item() * labels.size(0)
                _, preds = torch.max(logits, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                
        # Print Epoch Stats
        if (epoch + 1) % 5 == 0 or epoch == 0:
            t_acc = (train_correct / train_total) * 100
            v_acc = (val_correct / val_total) * 100
            v_loss = val_loss / val_total
            print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Acc: {t_acc:.1f}% | Val Acc: {v_acc:.1f}% | Val Loss: {v_loss:.4f}")
            
    return model

In [7]:
# gt_path = '/content/drive/MyDrive/pheno_crop/ground_truth.csv'
# s1_dir = '/content/drive/MyDrive/pheno_crop/sentinel_1'
# s2_dir = '/content/drive/MyDrive/pheno_crop/sentinel_2'

# all_plots = list(range(2, 54))

# full_dataset = ProductionPhenologyDataset(gt_path, s1_dir, s2_dir, plot_ids=all_plots)

# final_model = train_production_model(full_dataset, epochs=100, batch_size=32)